# 04 — Data Joining

**Research question:** now that 03 has standardized and cleaned D1/D2/D3 independently, how do they combine into one analysis-ready table? D1 (Billboard chart history) is the base table; D2 (Spotify audio features) is the primary join; D3 (lyric themes/genre) is an auxiliary join layered on top.

**Structurally different from 01-03.** Those stages apply one piece of logic to 3 datasets in a loop. 04 is two asymmetric join operations — D1 stays the base table throughout, D2 and D3 each get joined onto it once, not three identical passes. No new cleaning judgment happens here either (that's 02/03's job) — this stage only executes the merge and reports what happened.

## Architecture

| | |
|---|---|
| **CONFIG** | `JOIN_STAGES` — for each of the two join stages: which columns to carry over, and which column signals a successful match |
| **Engine** | One function, `dedup_and_left_join()` — dedup the right-hand table, left-join, report the match rate. Both join stages call the same function. |
| **Steps 1-6** | Load 03's output → D1+D2 primary join → match-rate evaluation (overall + by decade) → unmatched-sample check → D1+D2+D3 auxiliary join + full validation → save |

**Scope note:** mirrors R's join logic exactly (D1 as base table, D2 primary join, D3 auxiliary join) — nothing added or removed in scope, just the shared "dedup + left join + match rate" logic pulled into one function instead of R's two separately hand-written blocks.

## Step Blueprint

| Step | What It Does | Question Answered |
|------|------|------|
| **Step 1** | Load `wrangled_D1/D2/D3.pkl` from 03 | What does 03's output actually look like? |
| **Step 2** | D1+D2 Primary Join (`dedup_and_left_join`) | Attach D2's audio features (danceability, energy...) + target + decade onto every D1 chart row |
| **Step 3** | Match-Rate Evaluation | What fraction of D1 actually found a match in D2? Does that rate vary by decade? |
| **Step 4** | Unmatched-Sample Check | What do the unmatched rows actually look like — a data problem, or genuinely no overlap? |
| **Step 5** | D1+D2+D3 Auxiliary Join + Validation | Attach D3's genre/lyric-theme data too — what's the match rate, and how many rows matched both D2 and D3? |
| **Step 6** | Save | Persist the final analysis-ready table for 05 (EDA) / 06 (modeling) to pick up |

In [1]:
import pandas as pd

# Load wrangled datasets from Stage 03
d1 = pd.read_pickle('../Data/wrangled_D1.pkl')
d2 = pd.read_pickle('../Data/wrangled_D2.pkl')
d3 = pd.read_pickle('../Data/wrangled_D3.pkl')

print('[STEP 1] Load Wrangled Data')
print(f'D1 (Billboard): {len(d1):,} rows, {d1["join_key"].nunique():,} unique join_key')
print(f'D2 (Spotify):   {len(d2):,} rows, {d2["join_key"].nunique():,} unique join_key')
print(f'D3 (Music):     {len(d3):,} rows, {d3["join_key"].nunique():,} unique join_key')

[STEP 1] Load Wrangled Data
D1 (Billboard): 330,087 rows, 29,671 unique join_key
D2 (Spotify):   41,106 rows, 39,851 unique join_key
D3 (Music):     28,372 rows, 28,342 unique join_key


## CONFIG — `JOIN_STAGES`

**A real critical-thinking case, not a guess:** D3 happens to have columns also named `danceability`, `loudness`, `acousticness`, `instrumentalness`, `valence`, `energy` — same names as D2's Spotify audio features, but a completely different source and computation. R's original `select()` in Part 4 only pulls in 12 D3 columns (genre + 11 lyric-theme columns), deliberately excluding these name-colliding ones. Verified this was intentional, not an oversight — including both would force `left_join` to auto-suffix `_x`/`_y`, silently mixing two differently-sourced audio-feature sets under near-identical names, an easy source of misuse later. **Ported R's exact column selection here — only D3's 12 non-colliding columns get carried over.**

**Another decision ported faithfully, not "corrected":** both D2 and D3 are deduplicated on `join_key` before joining (`distinct(join_key, .keep_all=TRUE)` in R, `drop_duplicates()` here) — which keeps whichever row happens to appear first, not necessarily the "best" one by any rule. D2 has 41,106 rows but only 39,794 unique keys, meaning roughly 1,300 duplicate keys get resolved arbitrarily. This is an existing limitation in R's original logic, not something introduced in translation — ported as-is; worth revisiting only if this arbitrary tie-break turns out to actually distort a downstream result.

In [2]:
# ============================================================
# CONFIG -- Only edit this section when changing what each join carries over
# ============================================================

JOIN_STAGES = {
    "D1_D2": {
        "right_df": d2,
        "right_name": "D2",
        # Spotify audio features + target label + decade (all from D2, no name collisions with D1)
        "carry_columns": [
            "danceability", "energy", "key", "loudness", "mode",
            "speechiness", "acousticness", "instrumentalness",
            "liveness", "valence", "tempo", "duration_ms",
            "time_signature", "chorus_hit", "sections",
            "target", "decade",
        ],
        "match_indicator_col": "target",  # non-null target = matched a D2 row
    },
    "D1D2_D3": {
        "right_df": d3,
        "right_name": "D3",
        # Only D3's 12 non-colliding columns -- deliberately excludes D3's own
        # danceability/energy/valence/etc, which share names with D2's audio
        # features but come from a different source. See CONFIG markdown above.
        "carry_columns": [
            "genre", "dating", "violence", "world/life", "night/time",
            "romantic", "communication", "obscene", "music",
            "sadness", "feelings", "topic",
        ],
        "match_indicator_col": "genre",  # non-null genre = matched a D3 row
    },
}

## Engine — No edits needed below this line

`dedup_and_left_join()` is the only join logic in this stage — both join stages (D1+D2, D1D2+D3) call the exact same function. It doesn't know which dataset it's joining, only "here's a right-hand table, the columns to carry over, and a key — follow it, then report the match rate."

Execution order:
1. Dedup the right-hand table on `join_key` (keeps the first occurrence — matches R's `distinct()` behavior)
2. Select `join_key` + the requested columns, left-join onto the base table
3. Use `match_indicator_col` to count successful matches, print the match rate

In [3]:
def dedup_and_left_join(left_df, right_df, right_name, carry_columns, match_indicator_col, join_key="join_key"):
    """Generic join step used by both D1+D2 and D1D2+D3 stages.
    Dedups the right-hand table on join_key (keeps first occurrence, matching
    R's distinct(join_key, .keep_all=TRUE) behaviour -- arbitrary tie-break,
    inherited from R, not something this function decides), left-joins the
    requested columns onto left_df, and reports the match rate."""

    before = len(right_df)
    right_dedup = right_df.drop_duplicates(subset=join_key, keep="first")
    print(f'  {right_name}: {before:,} rows -> {len(right_dedup):,} unique {join_key} (dropped {before - len(right_dedup):,} duplicates)')

    joined = left_df.merge(
        right_dedup[[join_key] + carry_columns],
        on=join_key,
        how="left",
    )

    total = len(joined)
    matched = joined[match_indicator_col].notna().sum()
    unmatched = total - matched
    print(f'  Total rows after join: {total:,}')
    print(f'  Matched:   {matched:,} rows ({round(matched / total * 100, 2)}%)')
    print(f'  Unmatched: {unmatched:,} rows ({round(unmatched / total * 100, 2)}%)')

    return joined

## Step 2 — D1+D2 Primary Join

In [4]:
print('[STEP 2] D1 + D2 Join (Primary Join)')

stage = JOIN_STAGES["D1_D2"]
d1_d2_joined = dedup_and_left_join(
    d1, stage["right_df"], stage["right_name"],
    stage["carry_columns"], stage["match_indicator_col"],
)

[STEP 2] D1 + D2 Join (Primary Join)
  D2: 41,106 rows -> 39,851 unique join_key (dropped 1,255 duplicates)
  Total rows after join: 330,087
  Matched:   264,478 rows (80.12%)
  Unmatched: 65,609 rows (19.88%)


## Step 3 — Match-Rate Evaluation: Overall + By Decade

**Two different "decades" here — worth being careful about the names:** the joined-in `decade` column comes from D2 (Spotify's own '60s'-'10s' labels); `decade_d1` here is computed separately from D1's own `date` column (the decade a song actually charted). R's original code keeps these separate and separately named too — kept the same naming here to avoid conflating them.

In [5]:
print('[STEP 3] D1+D2 Match Rate Evaluation')

total = len(d1_d2_joined)
matched = d1_d2_joined["target"].notna().sum()
unmatched = total - matched
print(f'Total records: {total:,} rows')
print(f'Matched:       {matched:,} rows ({round(matched / total * 100, 2)}%)')
print(f'Unmatched:     {unmatched:,} rows ({round(unmatched / total * 100, 2)}%)')

# Match rate by D1's own chart-date decade (decade_d1), NOT the D2-sourced 'decade' column
d1_d2_joined["decade_d1"] = (d1_d2_joined["date"].dt.year // 10 * 10).astype(str) + "s"

decade_summary = (
    d1_d2_joined.groupby("decade_d1")
    .agg(total=("join_key", "size"), matched=("target", lambda s: s.notna().sum()))
    .reset_index()
)
decade_summary["match_pct"] = (decade_summary["matched"] / decade_summary["total"] * 100).round(2)
decade_summary = decade_summary.sort_values("decade_d1")
print()
print(decade_summary.to_string(index=False))

[STEP 3] D1+D2 Match Rate Evaluation
Total records: 330,087 rows
Matched:       264,478 rows (80.12%)
Unmatched:     65,609 rows (19.88%)



decade_d1  total  matched  match_pct
    1950s   7400      489       6.61
    1960s  52100    36826      70.68
    1970s  52187    40933      78.44
    1980s  52200    44945      86.10
    1990s  52100    42576      81.72
    2000s  52200    48301      92.53
    2010s  52200    49156      94.17
    2020s   9700     1252      12.91


## Step 4 — D1-D2 Unmatched-Sample Check

In [6]:
print('[STEP 4] D1-D2 Unmatched Records Sample Check')

unmatched_sample = (
    # 1. keep only rows that did NOT match (target is NaN)
    d1_d2_joined[d1_d2_joined["target"].isna()]
    # 2. add a year column, extracted from the date column
    .assign(year=lambda df: df["date"].dt.year)
    # 3. dedup by artist + song + year (a song can chart for many weeks,
    #    dedup keeps each song showing up only once)
    .drop_duplicates(subset=["artist_clean", "song_clean", "year"])
    # 4. sort oldest to newest
    .sort_values("year")
    # 5. keep only the columns worth looking at: artist, song, year
    [["artist_clean", "song_clean", "year"]]
    .head(10)
)
# print without the row-index column
print(unmatched_sample.to_string(index=False))

[STEP 4] D1-D2 Unmatched Records Sample Check
                                        artist_clean                        song_clean  year
                                     frankie vaughan                              judy  1958
                                        the olympics  (i wanna) dance with the teacher  1958
                                       the four aces                 the world outside  1958
                                       jimmy clanton                      a part of me  1958
the tommy dorsey orchestra starring warren covington        i want to be happy cha cha  1958
                                           doris day                    tunnel of love  1958
                                         johnny cash                    all over again  1958
                               bernie lowe orchestra                 intermission riff  1958
                   johnny cash and the tennessee two i just thought you'd like to know  1958
                    litt

**Why did 1958 (the 1950s) only match 6.61%?**

Billboard Hot 100 didn't launch until August 1958 — D1 covers every week's chart from that date on. Spotify's catalog skews toward modern and contemporary music; a lot of pre-1960s material (the vinyl era, early rock/jazz/blues) either isn't on Spotify at all, or is missing full audio-feature data. With so few 1950s tracks in D2 to begin with, most D1 rows from that era simply have nothing to match against.

**Why did 2020 (the 2020s) only match 12.91%?**

D2 (the Spotify dataset) was collected around early 2020 — it has nothing released after that cutoff. D1 (the Billboard chart) keeps recording new weekly chart data well past 2020. Matching a dataset frozen at "early 2020" against a chart that keeps adding new songs after that point means most of the newer material simply isn't in D2 to match against.

## Step 5 — D1+D2+D3 Auxiliary Join + Full Validation

In [7]:
print('[STEP 5] D1 + D2 + D3 Join (Auxiliary Join)')

stage = JOIN_STAGES["D1D2_D3"]
d1_d2_d3_joined = dedup_and_left_join(
    d1_d2_joined, stage["right_df"], stage["right_name"],
    stage["carry_columns"], stage["match_indicator_col"],
)

print()
print('-- Full validation (matches R Part 4) --')
total = len(d1_d2_d3_joined)
d2_matched = d1_d2_d3_joined["target"].notna().sum()
d3_matched = d1_d2_d3_joined["genre"].notna().sum()
both_matched = (d1_d2_d3_joined["target"].notna() & d1_d2_d3_joined["genre"].notna()).sum()

print(f'Total rows:              {total:,}')
print(f'D2 matched (has target): {d2_matched:,} ({round(d2_matched / total * 100, 2)}%)')
print(f'D3 matched (has genre):  {d3_matched:,} ({round(d3_matched / total * 100, 2)}%)')
print(f'Both D2 + D3 matched:    {both_matched:,} ({round(both_matched / total * 100, 2)}%)')

print()
print('Note: D3 matched here (row-level, %) reads higher than 03 Step 5 D1<->D3 key-level',
      'overlap (10.63%) -- that number was computed on UNIQUE join_keys, this one counts every',
      'weekly chart row, so a song that matches D3 gets counted once per week it charted.')

[STEP 5] D1 + D2 + D3 Join (Auxiliary Join)
  D3: 28,372 rows -> 28,342 unique join_key (dropped 30 duplicates)


  Total rows after join: 330,087
  Matched:   46,028 rows (13.94%)
  Unmatched: 284,059 rows (86.06%)

-- Full validation (matches R Part 4) --
Total rows:              330,087
D2 matched (has target): 264,478 (80.12%)
D3 matched (has genre):  46,028 (13.94%)
Both D2 + D3 matched:    44,559 (13.5%)

Note: D3 matched here (row-level, %) reads higher than 03 Step 5 D1<->D3 key-level overlap (10.63%) -- that number was computed on UNIQUE join_keys, this one counts every weekly chart row, so a song that matches D3 gets counted once per week it charted.


## Step 6 — Save

Persisted to `.pkl` for 05 (EDA) / 06 (modeling) to pick up — same pattern as 01's Step 8 and 03's Step 6.

In [8]:
print('[STEP 6] Save Final Analysis Dataset')

save_path = r'..\Data\D1_D2_D3_joined.pkl'
d1_d2_d3_joined.to_pickle(save_path)

print(f'[OK] Saved to: {save_path}')
print(f'Total rows:    {len(d1_d2_d3_joined):,}')
print(f'Total columns: {d1_d2_d3_joined.shape[1]}')

[STEP 6] Save Final Analysis Dataset


[OK] Saved to: ..\Data\D1_D2_D3_joined.pkl
Total rows:    330,087
Total columns: 40
